<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_6_Pivot_Crosstab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.6 — Pivot Tables, Crosstabs, and Descriptive Tools

## Learning Objectives

By the end of this notebook you should be able to:

- Use `pd.crosstab()` to build frequency and proportion tables
- Use `pd.pivot_table()` to summarize a numeric column across two categorical dimensions
- Add row/column totals with `margins=True`
- Use `.corr()` to find pairwise correlations
- Use `.nlargest()` and `.nsmallest()` to rank rows
- Choose between `groupby`, `crosstab`, and `pivot_table` for a given problem

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
print(df.shape)
df.head()

## 1. `pd.crosstab()` — Frequency Tables

`pd.crosstab()` counts how many observations fall into each combination of two (or more) categorical variables. The result is sometimes called a **contingency table** or **cross-tabulation**.

In [ ]:
# How many passengers of each sex survived vs did not survive?
pd.crosstab(df["sex"], df["survived"])

In [ ]:
# More descriptive labels
pd.crosstab(
    df["sex"],
    df["survived"],
    rownames=["Sex"],
    colnames=["Survived (0=No, 1=Yes)"]
)

In [ ]:
# Passenger class vs survived
pd.crosstab(df["pclass"], df["survived"])

### Adding Margins (Totals)

`margins=True` adds row and column totals.

In [ ]:
pd.crosstab(df["pclass"], df["survived"], margins=True, margins_name="Total")

### Normalizing to Proportions

The `normalize` parameter converts counts to proportions:

| Value | Meaning |
|---|---|
| `"index"` | Proportions within each **row** (each row sums to 1) |
| `"columns"` | Proportions within each **column** (each column sums to 1) |
| `"all"` | Proportions of the full table (all cells sum to 1) |

In [ ]:
# What fraction of each sex survived?
# normalize='index': each row (sex) sums to 1
pd.crosstab(df["sex"], df["survived"], normalize="index").round(2)

In [ ]:
# Of all survivors, what fraction were male vs female?
# normalize='columns': each column sums to 1
pd.crosstab(df["sex"], df["survived"], normalize="columns").round(2)

In [ ]:
# Class vs survived, normalized by row, with totals
pd.crosstab(
    df["pclass"], df["survived"],
    rownames=["Class"],
    colnames=["Survived"],
    normalize="index",
    margins=True
).round(2)

## 2. `pd.pivot_table()` — Aggregated Summary Tables

`pd.pivot_table()` is more powerful than `crosstab`. Instead of just counting, it can aggregate *any numeric column* using *any function*. Think of it as a `.groupby()` whose result is automatically shaped into a 2-D grid.

Key parameters:
- `values` — the column to aggregate
- `index` — variable(s) that become **row** labels
- `columns` — variable(s) that become **column** labels
- `aggfunc` — the aggregation function(s) (default: `"mean"`)

In [ ]:
# Average survival rate by class (rows) and sex (columns)
pd.pivot_table(
    df,
    values="survived",
    index="pclass",
    columns="sex",
    aggfunc="mean",
).round(2)

In [ ]:
# Average fare by class and sex
pd.pivot_table(
    df,
    values="fare",
    index="pclass",
    columns="sex",
    aggfunc="mean",
).round(2)

### Adding Margins

In [ ]:
pd.pivot_table(
    df,
    values="survived",
    index="pclass",
    columns="sex",
    aggfunc="mean",
    margins=True,
    margins_name="Overall",
).round(2)

### Multiple Aggregation Functions

In [ ]:
pd.pivot_table(
    df,
    values="fare",
    index="pclass",
    columns="sex",
    aggfunc=["mean", "count"],
).round(2)

## 3. GroupBy vs Crosstab vs Pivot Table

| Tool | Best for |
|---|---|
| `.groupby().agg()` | Flexible aggregation; arbitrary output; result is a DataFrame you continue to work with |
| `pd.crosstab()` | **Counts and proportions** of two categorical variables; exploratory work |
| `pd.pivot_table()` | **Any aggregation** displayed as a 2-D grid of two categorical variables; report-ready |

The same question can often be answered by any of the three:

In [ ]:
# Average survival rate by pclass and sex — three equivalent approaches

# 1. groupby + unstack
r1 = df.groupby(["pclass","sex"])["survived"].mean().unstack("sex").round(2)

# 2. pivot_table
r2 = pd.pivot_table(df, values="survived", index="pclass",
                    columns="sex", aggfunc="mean").round(2)

print("groupby result:")
print(r1)
print()
print("pivot_table result:")
print(r2)

## 4. `.corr()` — Pairwise Correlations

`.corr()` computes the Pearson correlation between all pairs of numeric columns. Values range from -1 (perfect negative) to +1 (perfect positive); 0 means no linear relationship.

In [ ]:
df[["survived", "pclass", "age", "sibsp", "parch", "fare"]].corr().round(2)

A few observations:
- **`pclass` and `survived`** are negatively correlated: third-class passengers (pclass = 3) survived at lower rates.
- **`fare` and `survived`** are positively correlated: higher fares correspond to higher survival — but this is largely explained by class (first-class passengers paid more and survived more).
- Correlation measures *linear association*, not causation.

## 5. `.nlargest()` and `.nsmallest()`

Return the rows with the `n` largest or smallest values in a given column.

In [ ]:
# Top 5 most expensive fares
df.nlargest(5, "fare")[["name", "pclass", "fare", "survived"]]

In [ ]:
# 5 youngest passengers
df.nsmallest(5, "age")[["name", "pclass", "age", "survived"]]

## 6. Other Quick Descriptive Tools

In [ ]:
# .describe() on the full DataFrame (numerics only by default)
df.describe().round(2)

In [ ]:
# .value_counts() with normalize=True for proportions
df["sex"].value_counts(normalize=True).round(3)

In [ ]:
# .value_counts() on multiple columns at once using a loop
for col in ["pclass", "sex"]:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

## 7. A Complete Descriptive Profile

Putting it all together.

In [ ]:
sep = "═" * 50
print(sep)
print("TITANIC — DESCRIPTIVE SUMMARY")
print(sep)

print(f"\nPassengers: {len(df):,}")
print(f"Survived:   {df['survived'].sum():,} ({df['survived'].mean():.1%})")

print("\nSurvival rate by class:")
for cls, rate in df.groupby("pclass")["survived"].mean().items():
    print(f"  Class {cls}: {rate:.1%}")

print("\nSurvival rate by sex:")
for sex, rate in df.groupby("sex")["survived"].mean().items():
    print(f"  {sex.capitalize()}: {rate:.1%}")

print("\nSurvival rate by class × sex:")
print(pd.pivot_table(df, values="survived", index="pclass",
                     columns="sex", aggfunc="mean").round(2).to_string())

print("\nAge summary:")
print(df["age"].describe().round(1).to_string())

print("\nFare summary:")
print(df["fare"].describe().round(2).to_string())

## Your Turn

1. Use `pd.crosstab()` to examine survival by passenger class. Normalize by row. Which class had the highest survival rate?
2. Build a pivot table showing the **median fare**, with passenger class as rows and sex as columns. Add margins.
3. Among the 10 passengers who paid the highest fares, what fraction survived?
4. Compute the correlation between `age` and `fare`. Is it positive or negative? What does that suggest?
5. Use `pd.crosstab()` with `normalize="all"` to find what percentage of all passengers were in each class/sex combination. Which combination was the most common?

In [ ]:
# Your code here